# Assignment 4 - Compare Baseline, SFT, and PPO on HH-RLHF

This notebook compares responses from three models on a small set of prompts extracted from HH-RLHF:
- the baseline model
- the SFT model
- the PPO (RLHF) model

It also uses the saved reward model to score each response, then saves tables for later reporting.


## Step 1: Install packages

Run this first in Colab. If Colab asks for a runtime restart, restart and run the notebook again.


In [ ]:
!pip -q install -U "transformers>=4.45.0" "datasets>=2.20.0" "accelerate>=0.33.0" \
                  "trl==0.29.0" "peft>=0.12.0" "bitsandbytes>=0.43.0" "sentencepiece" "pandas"

## Step 2: Mount Google Drive

This notebook expects the saved SFT, reward-model, and PPO outputs to already exist in Google Drive.


In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Google Drive is ready.')
except ImportError:
    print('This notebook is not running in Google Colab.')


## Step 3: Imports, paths, and options

Set the model family here. The default below uses the 1.5B runs.


In [ ]:
import json
import os
import random
from datetime import datetime

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from peft import AutoPeftModelForCausalLM, AutoPeftModelForSequenceClassification
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed

SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

IN_COLAB = "COLAB_GPU" in os.environ
DRIVE_ROOT = "/content/drive/MyDrive/RLHF4LLMs"
BASE_ROOT = DRIVE_ROOT if IN_COLAB else os.path.abspath(".")

MODEL_SIZE = "1.5B"  # choose from: "0.5B", "1.5B"
NUM_PROMPTS = 20
MAX_PROMPT_LENGTH = 256
MAX_NEW_TOKENS = 128
PROMPT_DATASET = "Anthropic/hh-rlhf"

if MODEL_SIZE == "0.5B":
    BASE_MODEL_NAME = "Qwen/Qwen2.5-0.5B"
    SFT_RUN_NAME = "sft_baseline_lora"
    RM_RUN_NAME = "reward_model_hh_rlhf_lora"
    PPO_RUN_NAME = "ppo_minimal_lora"
elif MODEL_SIZE == "1.5B":
    BASE_MODEL_NAME = "Qwen/Qwen2.5-1.5B"
    SFT_RUN_NAME = "sft_baseline_qwen25_1p5b_lora"
    RM_RUN_NAME = "reward_model_hh_rlhf_qwen25_1p5b_lora"
    PPO_RUN_NAME = "ppo_minimal_qwen25_1p5b_lora"
else:
    raise ValueError("MODEL_SIZE must be '0.5B' or '1.5B'.")

model_size_tag = MODEL_SIZE.lower().replace('.', 'p')
RESULTS_RUN_NAME = f"compare_results_{model_size_tag}_{SFT_RUN_NAME}_vs_{RM_RUN_NAME}"
RESULTS_FILE_STEM = f"compare_{model_size_tag}_{SFT_RUN_NAME}_vs_{RM_RUN_NAME}"

RESULTS_DIR = os.path.join(BASE_ROOT, RESULTS_RUN_NAME)
SFT_MODEL_DIR = os.path.join(BASE_ROOT, SFT_RUN_NAME, "final_model")
REWARD_MODEL_DIR = os.path.join(BASE_ROOT, RM_RUN_NAME, "final_model")
PPO_MODEL_DIR = os.path.join(BASE_ROOT, PPO_RUN_NAME, "final_model")

os.makedirs(RESULTS_DIR, exist_ok=True)

USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
MODEL_DTYPE = torch.bfloat16 if USE_BF16 else (torch.float16 if torch.cuda.is_available() else torch.float32)

print("MODEL_SIZE:", MODEL_SIZE)
print("BASE_MODEL_NAME:", BASE_MODEL_NAME)
print("SFT_MODEL_DIR:", SFT_MODEL_DIR)
print("REWARD_MODEL_DIR:", REWARD_MODEL_DIR)
print("PPO_MODEL_DIR:", PPO_MODEL_DIR)
print("RESULTS_RUN_NAME:", RESULTS_RUN_NAME)
print("RESULTS_DIR:", RESULTS_DIR)
print("MODEL_DTYPE:", MODEL_DTYPE)

## Step 4: Check prerequisite model files

The comparison needs the saved SFT, reward-model, and PPO adapters.


In [ ]:
required_files = [
    os.path.join(SFT_MODEL_DIR, "adapter_config.json"),
    os.path.join(REWARD_MODEL_DIR, "adapter_config.json"),
    os.path.join(PPO_MODEL_DIR, "adapter_config.json"),
]

missing = [path for path in required_files if not os.path.exists(path)]
if missing:
    raise FileNotFoundError(
        "Missing required adapter files. Missing:\n" + "\n".join(missing)
    )

print("All required model files were found.")

## Step 5: Load tokenizer and extract 20 prompts from HH-RLHF

We take prompt-style conversation prefixes from HH-RLHF and reuse the same 20 prompts for all three models.

> Note: HH-RLHF contains harmful, unsafe, and offensive content by design.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

raw_hh = load_dataset(PROMPT_DATASET, split="test")

def extract_prompt_from_hh(text):
    text = text.strip()
    marker = "\n\nAssistant:"
    if marker in text:
        prompt, _, _ = text.rpartition(marker)
        prompt = (prompt + marker).strip()
    else:
        prompt = text
    return prompt

prompt_records = []
seen = set()

for ex in raw_hh:
    chosen = ex["chosen"].strip()
    rejected = ex["rejected"].strip()
    prompt_from_chosen = extract_prompt_from_hh(chosen)
    prompt_from_rejected = extract_prompt_from_hh(rejected)
    prompt = prompt_from_chosen if len(prompt_from_chosen) >= len(prompt_from_rejected) else prompt_from_rejected

    if prompt and prompt not in seen:
        seen.add(prompt)
        prompt_records.append({"prompt_id": len(prompt_records) + 1, "prompt": prompt})

    if len(prompt_records) >= NUM_PROMPTS:
        break

if len(prompt_records) < NUM_PROMPTS:
    raise ValueError(f"Only found {len(prompt_records)} prompts, expected {NUM_PROMPTS}.")

prompts_df = pd.DataFrame(prompt_records)
prompts_df.head(3)

## Step 6: Load baseline, SFT, PPO, and reward models

The baseline is the raw causal LM. SFT and PPO are loaded from saved adapters. The reward model stays frozen and is used only for scoring.


In [ ]:
baseline_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    torch_dtype=MODEL_DTYPE,
    device_map="auto",
)
baseline_model.config.pad_token_id = tokenizer.pad_token_id
baseline_model.eval()

sft_model = AutoPeftModelForCausalLM.from_pretrained(
    SFT_MODEL_DIR,
    is_trainable=False,
    torch_dtype=MODEL_DTYPE,
    low_cpu_mem_usage=True,
)
sft_model.config.pad_token_id = tokenizer.pad_token_id
sft_model.eval()

ppo_model = AutoPeftModelForCausalLM.from_pretrained(
    PPO_MODEL_DIR,
    is_trainable=False,
    torch_dtype=MODEL_DTYPE,
    low_cpu_mem_usage=True,
)
ppo_model.config.pad_token_id = tokenizer.pad_token_id
ppo_model.eval()

reward_model = AutoPeftModelForSequenceClassification.from_pretrained(
    REWARD_MODEL_DIR,
    is_trainable=False,
    num_labels=1,
    torch_dtype=MODEL_DTYPE,
    low_cpu_mem_usage=True,
)
if torch.cuda.is_available():
    reward_model = reward_model.to("cuda")
reward_model.config.pad_token_id = tokenizer.pad_token_id
reward_model.config.use_cache = False
reward_model.eval()

print("Models loaded.")
print("Reward model device:", next(reward_model.parameters()).device)

## Step 7: Generate responses and score them

We use greedy decoding for a stable side-by-side comparison. Each response is then scored by the saved reward model.


In [ ]:
def clean_assistant_response(text):
    cleaned = text.strip()
    stop_markers = ["\n\nHuman:", "\nHuman:", "Human:"]
    for marker in stop_markers:
        if marker in cleaned:
            cleaned = cleaned.split(marker)[0].strip()
    return cleaned

def generate_response(model, prompt, max_prompt_length=MAX_PROMPT_LENGTH, max_new_tokens=MAX_NEW_TOKENS):
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=max_prompt_length,
        padding=False,
    )
    model_device = next(model.parameters()).device
    inputs = {k: v.to(model_device) for k, v in inputs.items()}

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    generated = output[0, inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(generated, skip_special_tokens=True).strip()
    return clean_assistant_response(response)

def reward_score(prompt, response, max_length=512):
    text = prompt + response
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=max_length,
        padding=False,
    )
    reward_device = "cuda" if torch.cuda.is_available() else next(reward_model.parameters()).device
    inputs = {k: v.to(reward_device) for k, v in inputs.items()}
    with torch.no_grad():
        logits = reward_model(**inputs).logits
    return float(logits.squeeze().float().cpu().item())

rows = []
for item in prompt_records:
    prompt_id = item["prompt_id"]
    prompt = item["prompt"]

    baseline_response = generate_response(baseline_model, prompt)
    sft_response = generate_response(sft_model, prompt)
    ppo_response = generate_response(ppo_model, prompt)

    baseline_reward = reward_score(prompt, baseline_response)
    sft_reward = reward_score(prompt, sft_response)
    ppo_reward = reward_score(prompt, ppo_response)

    winner = max(
        [("baseline", baseline_reward), ("sft", sft_reward), ("ppo", ppo_reward)],
        key=lambda x: x[1],
    )[0]

    rows.append({
        "prompt_id": prompt_id,
        "prompt": prompt,
        "baseline_response": baseline_response,
        "sft_response": sft_response,
        "ppo_response": ppo_response,
        "baseline_reward": round(baseline_reward, 4),
        "sft_reward": round(sft_reward, 4),
        "ppo_reward": round(ppo_reward, 4),
        "winner_by_reward": winner,
    })

comparison_df = pd.DataFrame(rows)
comparison_df.head(3)

## Step 8: Show the full comparison table

This table is the main side-by-side result view for your report or discussion.


In [ ]:
pd.set_option("display.max_colwidth", 200)
display(comparison_df)

## Step 9: Summarize the comparison

This gives a compact quantitative view of how often each model wins under the reward model and what the average reward looks like.


In [ ]:
summary = {
    "model_size": MODEL_SIZE,
    "num_prompts": len(comparison_df),
    "baseline_avg_reward": round(float(comparison_df["baseline_reward"].mean()), 4),
    "sft_avg_reward": round(float(comparison_df["sft_reward"].mean()), 4),
    "ppo_avg_reward": round(float(comparison_df["ppo_reward"].mean()), 4),
    "baseline_wins": int((comparison_df["winner_by_reward"] == "baseline").sum()),
    "sft_wins": int((comparison_df["winner_by_reward"] == "sft").sum()),
    "ppo_wins": int((comparison_df["winner_by_reward"] == "ppo").sum()),
    "saved_at": datetime.now().isoformat(timespec="seconds"),
}

summary_df = pd.DataFrame([
    {"metric": "Average reward", "baseline": summary["baseline_avg_reward"], "sft": summary["sft_avg_reward"], "ppo": summary["ppo_avg_reward"]},
    {"metric": "Reward-model wins", "baseline": summary["baseline_wins"], "sft": summary["sft_wins"], "ppo": summary["ppo_wins"]},
])

display(summary_df)
print(summary)

## Step 10: Save the tables for later use

This writes the detailed comparison and the summary to disk so you can reuse them in your report.


In [ ]:
comparison_csv = os.path.join(RESULTS_DIR, f"{RESULTS_FILE_STEM}.csv")
comparison_json = os.path.join(RESULTS_DIR, f"{RESULTS_FILE_STEM}.json")
summary_json = os.path.join(RESULTS_DIR, f"{RESULTS_FILE_STEM}_summary.json")

comparison_df.to_csv(comparison_csv, index=False, encoding="utf-8")
comparison_df.to_json(comparison_json, orient="records", force_ascii=False, indent=2)
with open(summary_json, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("Saved detailed comparison to:", comparison_csv)
print("Saved detailed comparison JSON to:", comparison_json)
print("Saved summary to:", summary_json)